<b>Código principal para cargar datos.</b>
<p> Configurar fechas de carga, seriales de boyas y rutas para archivos en \Configs\configuracion_general.py </p>


In [1]:
####################### N O  T O C A R ############################################
%load_ext autoreload
%autoreload 2
import sys
import os

# Agrega la ruta raíz del proyecto si no está
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# import Configs.configuracion_general as config
import services.Carga.cargar_datos_csv as carga 
from services.Utils.utilidades import *
from services.Correctores.corrector_utils import *
from configs.manager_diccionario_variables import * 
##################################################################################

### Paso 1. Cargar los datos de la sonda y de despliegue

In [2]:
rutas_de_sondas, seriales_encontrados = carga.buscar_nombre_de_archivo_de_sonda()

In [3]:
# rutas_de_sondas

In [4]:
diccionario_de_datos_de_sondas = carga.cargar_datos_de_sonda(rutas_de_sondas, seriales_encontrados)

Datos cargados correctamente para la sonda: 4876179
Datos cargados correctamente para la sonda: 4912197
Datos cargados correctamente para la sonda: 4887980
Datos cargados correctamente para la sonda: 4866660


In [5]:
# diccionario_de_datos_de_sondas["4878219"][0:40]

In [6]:
carga.agregar_coordenadas_de_despliegue_maniobras_al_excel_de_despliegue(diccionario_de_datos_de_sondas)

Sonda 4876179: Latitud maniobra: 19.008504, Longitud maniobra: -94.247739, Fecha y hora de despliegue maniobra: 2026-03-26 17:21:00
Sonda 4912197: Latitud maniobra: 18.830266, Longitud maniobra: -94.383395, Fecha y hora de despliegue maniobra: 2026-03-26 17:16:00
Sonda 4887980: Latitud maniobra: 18.865435, Longitud maniobra: -94.191263, Fecha y hora de despliegue maniobra: 2026-03-26 17:57:00
Sonda 4866660: Latitud maniobra: 18.622899, Longitud maniobra: -94.37891, Fecha y hora de despliegue maniobra: 2026-03-26 19:01:00


In [7]:
# diccionario_de_datos_de_sondas

### Paso 2. Seleccionar los datos dentro del rango de fechas del estudio, prepararlos y guardarlos

In [8]:
diccionario_de_sondas_en_fechas = carga.seleccionar_rango_de_fechas(diccionario_de_datos_de_sondas)
datos_de_sondas_sin_duplicados = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas)
datos_ordenados = carga.ordernar_datos_por_fecha(datos_de_sondas_sin_duplicados)

Datos ordenados por fecha para la sonda 4876179.
Datos ordenados por fecha para la sonda 4912197.
Datos ordenados por fecha para la sonda 4887980.
Datos ordenados por fecha para la sonda 4866660.


In [9]:
# diccionario_de_sondas_en_fechas["4866660"].head()

In [10]:
datos_con_tspan_redondeado = carga.crear_tspan_redondeado(datos_ordenados) # diccionario con tspan redondeado
datos_sin_repetidos = carga.existen_fechas_redondeadas_duplicadas(datos_con_tspan_redondeado) # Verifica si existen fechas redondeadas duplicadas, si existen lanza un error

Las siguientes fechas redondeadas están duplicadas en la sonda 4876179.
         tspan_de_envio       tspan_rounded
5   2026-03-26 21:29:00 2026-03-26 21:00:00
26  2026-03-27 16:59:00 2026-03-27 16:30:00
30  2026-03-27 19:46:00 2026-03-27 19:30:00
45  2026-03-28 06:25:00 2026-03-28 06:00:00
61  2026-03-28 22:22:00 2026-03-28 22:00:00
99  2026-03-30 04:25:00 2026-03-30 04:00:00
102 2026-03-30 05:52:00 2026-03-30 05:30:00
118 2026-03-30 19:25:00 2026-03-30 19:00:00
122 2026-03-30 21:29:00 2026-03-30 21:00:00
138 2026-03-31 09:56:00 2026-03-31 09:30:00
150 2026-03-31 18:25:00 2026-03-31 18:00:00
152 2026-03-31 20:21:00 2026-03-31 20:00:00
actualmente se deja solo la primera ocurrencia, se eliminan las demás. Se suguiere hacer una funcion que distribuya a la hora anterior y/o siguiente.
Las siguientes fechas redondeadas están duplicadas en la sonda 4912197.
         tspan_de_envio       tspan_rounded
15  2026-03-27 00:59:00 2026-03-27 00:30:00
17  2026-03-27 01:57:00 2026-03-27 01:30:00
40

In [11]:
# datos_con_tspan_redondeado["4866660"].head()

In [12]:
# datos_sin_repetidos # Muestra de los datos de la primera sonda cualquiera

In [13]:
# A continuación se crea el dataframe vacío con todas las fechas que pueden existir y se combina con el dataframe de cada sonda
columnas = carga.get_cols_names_sin_tspan(diccionario = datos_sin_repetidos)
diccionario_con_dfs_vacios = carga.crear_diccionario_con_dataframes_vacios(diccionario= datos_sin_repetidos, keys=columnas)
datos_finales = carga.merge_df_vacio_con_datos(datos_sin_repetidos, diccionario_con_dfs_vacios) #datos finales realmente son los datos del periodo de vigencia del estudio

In [14]:
# datos_finales["4866660"].head()

In [15]:
# datos_finales["4878205"].head(10) # Ejemplo de los datos de salida

In [16]:
# Eliminar datos espurios (solo se revisa si hay valores de rapidez superiores a 2 m/s y se elimina toda la fila)
datos_finales = eliminar_datos_espurios(datos_finales)

Sonda 4887980: Se encontraron 1 filas con rapidez > 2 m/s. Valores reemplazados con NaN.


In [17]:
# Agregar componentes de la velocidad al diccionario con los dataframe de cada sonda
datos_finales = carga.agregar_componentes_de_la_velocidad(datos_finales)

In [18]:
# datos_finales["4866660"]

### Paso 3. Seleccionar todos los datos antes de la fecha de estudio

In [19]:
# En este caso estoy buscando las fechas desde la instalación hasta la fecha de inicio del análsisis
diccionario_de_sondas_en_fechas_anteriores = carga.seleccionar_rango_de_fechas(diccionario = diccionario_de_datos_de_sondas, 
                                                                    buscar_fechas_anteriores_al_estudio = True)
datos_de_sondas_sin_duplicados_anteriores = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas_anteriores)
datos_ordenados_anteriores = carga.ordernar_datos_por_fecha(datos_de_sondas_sin_duplicados_anteriores)
datos_con_tspan_redondeado_anteriores = carga.crear_tspan_redondeado(datos_ordenados_anteriores) # diccionario con tspan redondeado
datos_sin_repetidos_anteriores = carga.existen_fechas_redondeadas_duplicadas(datos_con_tspan_redondeado_anteriores) # Verifica si existen fechas redondeadas duplicadas, si existen lanza un error
# A continuación se crea el dataframe vacío con todas las fechas que pueden existir y se combina con el dataframe de cada sonda
columnas = carga.get_cols_names_sin_tspan(diccionario = datos_sin_repetidos_anteriores)
diccionario_con_dfs_vacios = carga.crear_diccionario_con_dataframes_vacios(diccionario= datos_sin_repetidos_anteriores, keys=columnas)
datos_anteriores = carga.merge_df_vacio_con_datos(datos_sin_repetidos_anteriores, diccionario_con_dfs_vacios)
datos_anteriores = eliminar_datos_espurios(datos_anteriores)
datos_anteriores = carga.agregar_componentes_de_la_velocidad(datos_anteriores)

Datos ordenados por fecha para la sonda 4876179.
Datos ordenados por fecha para la sonda 4912197.
Datos ordenados por fecha para la sonda 4887980.
Datos ordenados por fecha para la sonda 4866660.
No se encontraron fechas redondeadas duplicadas en la sonda 4876179.
No se encontraron fechas redondeadas duplicadas en la sonda 4912197.
No se encontraron fechas redondeadas duplicadas en la sonda 4887980.
No se encontraron fechas redondeadas duplicadas en la sonda 4866660.
El dataframe de la sonda 4876179 está vacío. Avanzando a la siguiente sonda
El dataframe de la sonda 4912197 está vacío. Avanzando a la siguiente sonda
El dataframe de la sonda 4887980 está vacío. Avanzando a la siguiente sonda
El dataframe de la sonda 4866660 está vacío. Avanzando a la siguiente sonda


### Paso 4. Eliminar NaNs de los datos de las sondas.
1. Eliminar NaNs del inicio de los datos.
- si la sonda tiene mediciones anteriores a la fecha de estudio, se eliminan esos.
- si la sonda comienza a medir en el mes del estudio, se eliminan esos datos inciales.
2. Eliminar NaNs del final de los datos. (Solo se eliminan los datos del final del periodo de estudio)

In [20]:
datos_anteriores

{}

In [21]:
# Eliminar NaNs iniciales y finales del dataframe de cada sonda; almanecnar esa información en el excel de despliegue
datos_antes_del_estudio, datos_del_estudio = carga.eliminar_nans_iniciales(datos_anteriores, datos_finales)
datos_del_estudio = carga.eliminar_nans_finales(datos_del_estudio)

In [22]:
# datos_anteriores["4878218"].head()

In [23]:
# datos_antes_del_estudio["4878218"].head()

### Paso 5. Guardar datos procesados en formato pkl

In [24]:
# Guardar datos anteriores a la fecha de estudio en archivo .pkl
carpeta_de_destino = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
nombre_de_archivo = get_nombre_del_archivo_de_datos_previos_a_la_fecha_de_estudio()
guardar_diccionario_como_pickle(diccionario = datos_antes_del_estudio, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)

# Guardar datos del estudio en archivo .pkl
carpeta_de_destino = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
nombre_de_archivo = get_nombre_archivo_datos_procesados()
guardar_diccionario_como_pickle(diccionario = datos_del_estudio, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)

El diccionario está vacío. No se guardará el archivo datos_previos_al_estudio.
Diccionario guardado correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202603\datos_procesados_sondas_oceanograficas.pkl


In [25]:
# datos_del_estudio["4878503"]